# TV Series Score Models

## Setup and configuration

In [ ]:
import re, warnings, time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
warnings.filterwarnings("ignore")

DATA_PATH = None
TARGET_LIST = ["combined_score", "imdb_rating", "tmdb_rating"]
MAIN_TARGET = "combined_score"
MAIN_FEATURE_SET = "new_people"
USE_START_YEAR = False
TRAIN_END_YEAR = 2022
MIN_CATEGORY_COUNT = 30
VAL_YEARS = [2019, 2020, 2021, 2022]
N_ITER = 12
SEED = 42
OUT_DIR = Path("model_output"); OUT_DIR.mkdir(exist_ok=True)

## Load series_dataset_full

In [ ]:
def find_csv():
    roots = [Path("/kaggle/input"), Path("output"), Path(".")]
    files = [p for r in roots if r.exists() for p in r.rglob("series_dataset_full.csv")]
    if not files:
        raise SystemExit("series_dataset_full.csv not found. Attach the Kaggle dataset, or set DATA_PATH.")
    return files[0]

DATA_PATH = Path(DATA_PATH) if DATA_PATH else find_csv()
full = pd.read_csv(DATA_PATH)
print(f"file: {DATA_PATH}")
print(f"series_dataset_full: {full.shape[0]} series x {full.shape[1]} columns")

## Build model-ready table

In [ ]:
def clean(v):
    return re.sub(r"[^0-9a-zA-Z_]+", "_", str(v)).strip("_")

def top_or_other(series):
    s_ = series.fillna("unknown")
    return s_.where(s_.map(s_.value_counts()) >= MIN_CATEGORY_COUNT, "other").map(clean)

def build_model_ready(full):
    df = full.copy()
    if "premiere_month" not in df and "premiere_date" in df:
        df["premiere_month"] = pd.to_datetime(df["premiere_date"], errors="coerce").dt.month
    if "is_coproduction" not in df and "origin_country" in df:
        df["is_coproduction"] = (df["origin_country"].fillna("").str.count(",") >= 1).astype(int)
    num = ["start_year", "premiere_month", "runtime_min", "s1_episode_count", "is_streaming_original",
           "is_adaptation", "is_true_story", "is_remake_or_reboot", "is_spinoff", "is_coproduction", "has_other_language",
           "creator_prior_avg_rating", "creator_prior_count", "creator_prior_best", "creator_is_debut",
           "cast_prior_avg_rating", "cast_prior_count", "cast_max_prior_rating"]
    num += [c for c in df.columns if c.startswith(("imdb_cast_prior_", "imdb_creative_prior_"))]
    missing = [c for c in num if c not in df.columns]
    if missing:
        print("not in this CSV (skipped):", missing)
    num = [c for c in num if c in df.columns]
    targets = [c for c in ["combined_score", "imdb_rating", "tmdb_rating", "rt_critic"] if c in df.columns]
    m = df[["imdb_id", "title"] + targets + num].copy()
    m = m.drop(columns=[c for c in m.columns if m[c].isna().all()])

    if "genres" in df:
        g = df["genres"].fillna("").str.split("|")
        counts = g.explode().value_counts()
        for name in counts[(counts >= MIN_CATEGORY_COUNT) & (counts.index != "")].index:
            m["genre_" + name.lower().replace("-", "_")] = g.apply(lambda x, n=name: int(n in x))
    cat = {}
    for c in ["title_type", "network_type"]:
        if c in df:
            cat[c] = df[c].fillna("unknown")
    if "age_rating" in df:
        a = df["age_rating"].fillna("unknown")
        cat["age_rating"] = a.where(a.isin(a.value_counts().head(6).index), "other")
    if "network" in df:
        cat["network_name"] = top_or_other(df["network"])
    m = pd.concat([m, pd.get_dummies(pd.DataFrame(cat), dtype=int)], axis=1)
    m["split"] = np.where(df["start_year"] <= TRAIN_END_YEAR, "train", "test")
    return m[m["combined_score"].notna()].reset_index(drop=True)

data = build_model_ready(full)
data.to_csv(OUT_DIR / "series_model_ready.csv", index=False)
print(f"model-ready table: {data.shape[0]} series x {data.shape[1]} columns -> {OUT_DIR / 'series_model_ready.csv'}")
print(f"XGBoost available: {HAS_XGB}")

## Feature sets

In [ ]:
ID_COLS = ["imdb_id", "title"]
TARGETS = [c for c in ["combined_score", "imdb_rating", "tmdb_rating", "rt_critic"] if c in data.columns]
TARGET_LIST = [t for t in TARGET_LIST if t in data.columns]
if MAIN_TARGET not in TARGET_LIST:
    raise SystemExit(f"MAIN_TARGET {MAIN_TARGET} must be one of {TARGET_LIST}")
drop = set(ID_COLS) | set(TARGETS) | {"split"} | (set() if USE_START_YEAR else {"start_year"})
ALL_FEATS = [c for c in data.columns if c not in drop]

OLD = [c for c in ["creator_prior_avg_rating", "creator_prior_count", "creator_prior_best", "creator_is_debut",
                   "cast_prior_avg_rating", "cast_prior_count", "cast_max_prior_rating"] if c in ALL_FEATS]
NEW = [c for c in ALL_FEATS if c.startswith(("imdb_cast_prior_", "imdb_creative_prior_"))]
BASE = [c for c in ALL_FEATS if c not in OLD and c not in NEW]
FEATURE_SETS = {"no_people": BASE, "old_people": BASE + OLD, "new_people": BASE + NEW, "both": BASE + OLD + NEW}
if not NEW:
    print("WARNING: no imdb_*_prior_* columns found. Re-run the dataset builder with section 6b.")
FEATS = FEATURE_SETS[MAIN_FEATURE_SET]

train = data[data["split"] == "train"].reset_index(drop=True)
holdout = data[data["split"] == "test"].reset_index(drop=True)
print(f"train: {len(train)} series ({train.start_year.min()}-{train.start_year.max()}) | hold-out: {len(holdout)} (kept aside)")
print(f"features: base {len(BASE)}, old people {len(OLD)}, new people {len(NEW)} | main set '{MAIN_FEATURE_SET}': {len(FEATS)}")
print("\nTargets (training series that have each rating):")
for t in TARGET_LIST:
    print(f"  {t:<15} {train[t].notna().sum():>5} series | mean {train[t].mean():.2f}, std {train[t].std():.2f}")

## Validation folds

In [ ]:
def year_folds(years, val_years):
    years = np.asarray(years)
    return [(np.where(years < v)[0], np.where(years == v)[0]) for v in val_years]

def metrics(y, p):
    return {"MAE": mean_absolute_error(y, p), "RMSE": mean_squared_error(y, p) ** 0.5,
            "R2": r2_score(y, p), "within_0.5": float(np.mean(np.abs(np.asarray(y) - p) <= 0.5))}

def cv_mae(make_est, X, y, folds, w=None, y_eval=None):
    errs = []
    for tr, va in folds:
        est = make_est()
        if w is None:
            est.fit(X.iloc[tr], y.iloc[tr])
        else:
            est.fit(X.iloc[tr], y.iloc[tr], sample_weight=w[tr])
        errs.append(mean_absolute_error((y if y_eval is None else y_eval).iloc[va], est.predict(X.iloc[va])))
    return float(np.mean(errs))
print("fold design:", [f"train < {v}, validate {v}" for v in VAL_YEARS])

## Model tuning

In [ ]:
def pipe(model, scale=True):
    steps = [("imp", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        steps.append(("sc", StandardScaler()))
    return Pipeline(steps + [("model", model)])

specs = {
    "Ridge": (pipe(Ridge()), {"model__alpha": [10, 30, 100, 300, 1000, 3000, 10000]}, "grid"),
    "kNN": (pipe(KNeighborsRegressor()), {"model__n_neighbors": [10, 20, 40, 80], "model__weights": ["uniform", "distance"]}, "grid"),
    "SVR": (pipe(SVR()), {"model__C": [0.1, 0.3, 1, 3, 10], "model__epsilon": [0.05, 0.1, 0.3],
                          "model__gamma": ["scale", 0.003, 0.01, 0.03]}, "grid"),
    "RandomForest": (pipe(RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=1), scale=False),
                     {"model__max_depth": [6, 10, None], "model__min_samples_leaf": [1, 2, 3, 5, 10],
                      "model__max_features": [0.2, 0.3, 0.5, 1.0]}, "random"),
    "HistGradientBoosting": (HistGradientBoostingRegressor(random_state=SEED),
                     {"learning_rate": [0.01, 0.02, 0.03, 0.05, 0.1], "max_depth": [2, 3, 5, None], "max_iter": [200, 400, 600],
                      "min_samples_leaf": [10, 20, 40], "l2_regularization": [0, 1, 10, 30]}, "random"),
}
if HAS_XGB:
    specs["XGBoost"] = (XGBRegressor(tree_method="hist", random_state=SEED, n_jobs=1, verbosity=0),
                        {"n_estimators": [300, 600], "max_depth": [3, 5, 7], "learning_rate": [0.03, 0.06, 0.1],
                         "subsample": [0.7, 1.0], "colsample_bytree": [0.6, 1.0], "min_child_weight": [1, 5]}, "random")

TUNED = {}
for target in TARGET_LIST:
    mask = train[target].notna().to_numpy()
    tr_t = train[mask].reset_index(drop=True)
    folds_t = year_folds(tr_t["start_year"], VAL_YEARS)
    X, y = tr_t[FEATS], tr_t[target]
    base_val = cv_mae(lambda: DummyRegressor(strategy="mean"), X, y, folds_t)
    print("=" * 78)
    print(f"TARGET: {target} | {len(tr_t)} training series | validation fold sizes {[len(v) for _, v in folds_t]}")
    print(f"Baseline (predict the training mean): validation MAE = {base_val:.3f}")
    searches, val_rows = {}, [{"model": "Baseline (train mean)", "val_MAE": base_val, "val_MAE_std": np.nan}]
    for name, (est, grid, kind) in specs.items():
        t0 = time.time()
        if kind == "grid":
            s = GridSearchCV(est, grid, cv=folds_t, scoring="neg_mean_absolute_error", n_jobs=-1)
        else:
            s = RandomizedSearchCV(est, grid, n_iter=N_ITER, cv=folds_t, scoring="neg_mean_absolute_error",
                                   n_jobs=-1, random_state=SEED)
        s.fit(X, y)
        searches[name] = s
        val_rows.append({"model": name, "val_MAE": -s.best_score_, "val_MAE_std": s.cv_results_["std_test_score"][s.best_index_]})
        print(f"  {name:<22} val MAE {-s.best_score_:.3f}  ({time.time() - t0:.0f}s)")
    val_table = pd.DataFrame(val_rows).sort_values("val_MAE").reset_index(drop=True)
    best_name = val_table[val_table["model"].isin(searches)].iloc[0]["model"]
    TUNED[target] = dict(searches=searches, val_table=val_table, best_name=best_name, base_val=base_val,
                         folds=folds_t, mask=mask, n_train=len(tr_t))
    print(f"  -> best on validation: {best_name}")
    print(val_table.round(3).to_string())

## Experiment: people features

In [ ]:
Xtr, ytr = train[FEATS], train[MAIN_TARGET]
folds = TUNED[MAIN_TARGET]["folds"]
hgb_params = TUNED[MAIN_TARGET]["searches"]["HistGradientBoosting"].best_params_
make_hgb = lambda: HistGradientBoostingRegressor(random_state=SEED, **hgb_params)

exp_a = pd.DataFrame([{"feature_set": k, "n_features": len(v), "val_MAE": cv_mae(make_hgb, train[v], ytr, folds)}
                      for k, v in FEATURE_SETS.items()]).sort_values("val_MAE")
exp_a["gain_vs_no_people"] = exp_a.loc[exp_a.feature_set == "no_people", "val_MAE"].iloc[0] - exp_a["val_MAE"]
print("4a. People features (validation MAE, lower is better)")
print(exp_a.round(3).to_string())

## Experiments: sample weights and TMDB threshold

In [ ]:
if not {"imdb_votes", "tmdb_votes", "tmdb_rating", "imdb_rating"} <= set(full.columns):
    print("Needs series_dataset_full.csv (with vote counts): skipping experiments 4b and 4c.")
else:
    f = train[["imdb_id"]].merge(full[["imdb_id", "imdb_rating", "imdb_votes", "tmdb_rating", "tmdb_votes"]],
                                 on="imdb_id", how="left")

    w = np.log1p(f["imdb_votes"].fillna(f["imdb_votes"].median())).to_numpy()
    exp_b = pd.DataFrame([{"training": "unweighted", "val_MAE": cv_mae(make_hgb, Xtr, ytr, folds)},
                          {"training": "weighted by log(1+IMDb votes)", "val_MAE": cv_mae(make_hgb, Xtr, ytr, folds, w=w)}])
    print("4b. Sample weights"); print(exp_b.round(3).to_string())

    def combined_from(d, tmdb_min):
        s = np.stack([d["imdb_rating"].to_numpy(float),
                      np.where(d["tmdb_votes"].fillna(0) >= tmdb_min, d["tmdb_rating"], np.nan)], axis=1)
        wt = np.stack([np.log1p(d["imdb_votes"].fillna(0)), np.log1p(d["tmdb_votes"].fillna(0))], axis=1)
        wt = np.where(np.isnan(s), 0, wt)
        return pd.Series((np.nan_to_num(s) * wt).sum(1) / wt.sum(1))

    y30 = combined_from(f, 30)
    print(f"sanity: rebuilt threshold-30 target differs from the file's target by {np.abs(y30 - ytr).max():.3f} (max abs, should be ~0)")
    rows = []
    for T in [30, 100, 300]:
        yT = combined_from(f, T)
        rows.append({"tmdb_min_votes": T,
                     "series_with_tmdb_rating_%": round(100 * (f["tmdb_votes"].fillna(0) >= T).mean(), 1),
                     "val_MAE_own_target": cv_mae(make_hgb, Xtr, yT, folds),
                     "val_MAE_vs_threshold30_target": cv_mae(make_hgb, Xtr, yT, folds, y_eval=y30)})
    print("4c. TMDB threshold"); print(pd.DataFrame(rows).round(3).to_string())

## Hold-out evaluation

In [ ]:
FINAL = {}
for target in TARGET_LIST:
    T = TUNED[target]
    ho = holdout[holdout[target].notna()]
    Xte, yte = ho[FEATS], ho[target]
    base_pred = np.full(len(yte), train.loc[T["mask"], target].mean())
    rows = [{"model": "Baseline (train mean)", **metrics(yte, base_pred)}]
    preds = {}
    for name, s in T["searches"].items():
        preds[name] = s.best_estimator_.predict(Xte)
        rows.append({"model": name, **metrics(yte, preds[name])})
    test_table = pd.DataFrame(rows).rename(columns={"MAE": "test_MAE", "RMSE": "test_RMSE", "R2": "test_R2"})
    results = T["val_table"].merge(test_table, on="model").sort_values("val_MAE").reset_index(drop=True)
    FINAL[target] = dict(results=results, preds=preds, ho=ho, yte=yte, best_pred=preds[T["best_name"]], base_test=test_table.iloc[0])
    print("=" * 78)
    print(f"{target}: val = year folds on 2015-2022, test = hold-out 2023-2025 ({len(ho)} series)")
    print(results.round(3).to_string())

summary = []
for target in TARGET_LIST:
    r = FINAL[target]["results"]; b = r[r["model"] == TUNED[target]["best_name"]].iloc[0]
    base = FINAL[target]["base_test"]
    summary.append({"target": target, "best_model (by validation)": b["model"], "n_test": len(FINAL[target]["ho"]),
                    "baseline_test_MAE": base["test_MAE"], "model_test_MAE": b["test_MAE"],
                    "improvement_%": 100 * (1 - b["test_MAE"] / base["test_MAE"]),
                    "test_R2": b["test_R2"], "within_0.5": b["within_0.5"]})
summary = pd.DataFrame(summary)
print("\n" + "=" * 78 + "\nSUMMARY: best model per target on the hold-out set")
print(summary.round(3).to_string(index=False))

## Evaluation plots

In [ ]:
n = len(TARGET_LIST)
fig, ax = plt.subplots(1, n, figsize=(5.2 * n, 4.2), squeeze=False)
for a, t in zip(ax[0], TARGET_LIST):
    r = FINAL[t]["results"].sort_values("test_MAE")
    a.barh(r["model"], r["test_MAE"], color="#4c78a8")
    a.axvline(FINAL[t]["base_test"]["test_MAE"], color="red", ls="--", label="baseline")
    a.set_title(t); a.set_xlabel("Test MAE (lower is better)"); a.legend()
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, n, figsize=(5.2 * n, 4.6), squeeze=False)
for a, t in zip(ax[0], TARGET_LIST):
    y_, p_ = FINAL[t]["yte"], FINAL[t]["best_pred"]
    a.scatter(y_, p_, s=10, alpha=0.5)
    lim = [min(y_.min(), p_.min()) - 0.2, max(y_.max(), p_.max()) + 0.2]
    a.plot(lim, lim, "k--", lw=1); a.set_xlim(lim); a.set_ylim(lim)
    a.set_xlabel("Actual"); a.set_ylabel("Predicted"); a.set_title(f"{t}: {TUNED[t]['best_name']}")
plt.tight_layout(); plt.show()

## Direct vs merged combined score

In [ ]:
if {"combined_score", "imdb_rating", "tmdb_rating"} <= set(TARGET_LIST):
    ho_c, yc = FINAL["combined_score"]["ho"], FINAL["combined_score"]["yte"]
    est_i = TUNED["imdb_rating"]["searches"][TUNED["imdb_rating"]["best_name"]].best_estimator_
    est_t = TUNED["tmdb_rating"]["searches"][TUNED["tmdb_rating"]["best_name"]].best_estimator_
    pi, pt = est_i.predict(ho_c[FEATS]), est_t.predict(ho_c[FEATS])
    b = train.dropna(subset=["imdb_rating", "tmdb_rating"])
    d1, d2 = b["imdb_rating"] - b["tmdb_rating"], b["combined_score"] - b["tmdb_rating"]
    w_imdb = float(np.clip((d1 * d2).sum() / (d1 ** 2).sum(), 0, 1))
    combo = w_imdb * pi + (1 - w_imdb) * pt
    print(f"fitted IMDb weight in the combined score: {w_imdb:.2f}")
    print(f"combined, predicted directly : test MAE {mean_absolute_error(yc, FINAL['combined_score']['best_pred']):.3f}")
    print(f"combined, from the two models: test MAE {mean_absolute_error(yc, combo):.3f}")
else:
    print("Needs all three targets in TARGET_LIST.")

## Permutation importance

In [ ]:
IMP = {}
for t in TARGET_LIST:
    est = TUNED[t]["searches"][TUNED[t]["best_name"]].best_estimator_
    ho = FINAL[t]["ho"]
    imp = permutation_importance(est, ho[FEATS], ho[t], n_repeats=10, random_state=SEED,
                                 scoring="neg_mean_absolute_error", n_jobs=-1)
    IMP[t] = (pd.DataFrame({"feature": FEATS, "importance": imp.importances_mean, "std": imp.importances_std})
                .sort_values("importance", ascending=False).reset_index(drop=True))
    print(f"\nTop 10 features for {t} ({TUNED[t]['best_name']}):")
    print(IMP[t].head(10).round(4).to_string(index=False))

fig, ax = plt.subplots(1, len(TARGET_LIST), figsize=(6 * len(TARGET_LIST), 5.5), squeeze=False)
for a, t in zip(ax[0], TARGET_LIST):
    top = IMP[t].head(12).iloc[::-1]
    a.barh(top["feature"], top["importance"], xerr=top["std"], color="#59a14f")
    a.set_title(t); a.set_xlabel("Increase in MAE when shuffled")
plt.tight_layout(); plt.show()

## Save results

In [ ]:
pd.concat([FINAL[t]["results"].assign(target=t) for t in TARGET_LIST]).to_csv(OUT_DIR / "model_comparison_all_targets.csv", index=False)
summary.to_csv(OUT_DIR / "summary_per_target.csv", index=False)
pd.concat([IMP[t].assign(target=t) for t in TARGET_LIST]).to_csv(OUT_DIR / "permutation_importance.csv", index=False)

pred_df = holdout[["imdb_id", "title", "start_year"]].copy()
for t in TARGET_LIST:
    pred_df[f"actual_{t}"] = holdout[t]
    pred_df.loc[FINAL[t]["ho"].index, f"pred_{t}"] = FINAL[t]["best_pred"]
pred_df.to_csv(OUT_DIR / "holdout_predictions.csv", index=False)

joblib.dump({t: {"model": TUNED[t]["searches"][TUNED[t]["best_name"]].best_estimator_, "features": FEATS,
                 "name": TUNED[t]["best_name"]} for t in TARGET_LIST}, OUT_DIR / "best_models.joblib")

def predict_new(new_df):
    bundles = joblib.load(OUT_DIR / "best_models.joblib")
    return pd.DataFrame({f"pred_{t}": b["model"].predict(new_df.reindex(columns=b["features"])) for t, b in bundles.items()},
                        index=new_df.index)

print("Saved to", OUT_DIR.resolve())
print(pred_df.round(2).head(8).to_string(index=False))